#### Data Skew
The Silent Pipeline Killer in Spark

#### What is Data Skew?
Data skew happens when a small number of partitions contain significantly more data than the others.

In a distributed system like Spark, work is supposed to be balanced across executors.

But when one partition becomes overloaded:
* One executor struggles with massive data.
* Other executors finish early and sit idle
* The entire pipeline slows down waiting for a few tasks to  complete

#### Some common approaches:
* Repartition data properly
* Salting skewed keys
* Using broadcast joins for smaller datasets
* Optimizing parition strategies
* Monitoring shuffle-heavy transformations.

##### Suppose We have a sals table where most transactions belong to a single customer.

In [0]:
#Sales Table(Fact)

sales_data=[(1001,500),(1001,200),(1001,300),(1002,150),(1003,250)]
sales_columns=["cust_id","sales_amt"]
sales_df = spark.createDataFrame(sales_data,sales_columns)
display(sales_df)


##### Customer_ID=1001 has many more records -> causes data skew during join with the customer dimension table.

In [0]:
#customer table
customer_data=[(1001,"Raj"),(1002,"Ravi"),(1003,"Rajesh")]
customer_columns=["cust_id","cust_name"]
customer_df = spark.createDataFrame(customer_data,customer_columns)
display(customer_df)

#### Apply Salting

###### Step1: Add salt to the large(fact) table

In [0]:
from pyspark.sql import functions as F
SALT_BUCKET=5
sales_salted = sales_df.withColumn("salt",F.floor(F.rand()*SALT_BUCKET))
display(sales_salted)


##### Step2: Replicate dimension table for all salt values

In [0]:
customer_salted = customer_df.withColumn("salt_array",F.array(*[F.lit(i) for i in range(SALT_BUCKET)])) \
    .withColumn("salt",F.explode(F.col("salt_array")))
display(customer_salted)


In [0]:
customer_salted=customer_salted.drop("salt_array")
display(customer_salted)

##### Step3: Join using(customer_id,salt)

In [0]:
joined_df = sales_salted.join(customer_salted,["cust_id","salt"],how='inner')
display(joined_df)

#### HOW IT WORKS(RESULT)

After salting, records for customer_id=1001 are spread across 5 salt buckets.

Sales Table after salting (SALT_BUCKET=5)

In [0]:
display(joined_df)

###### Load is distributed across 5 *partitions* instead of 1-> balanced workload, faster join and job completion.

#### Key Takeway

Data skew can silently become the biggest performance bottleneck in Spark.

Salting is a simple yet highly effective technique that significantly improves parallelism and reduces execution time in Databricks.

###